<a href="https://colab.research.google.com/github/sangjkim930/AI-Driven-Research-Methodology/blob/main/02_In_Memory_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Simple In-Memory RAG System

In this exercise, we will build a simple RAG system using academic papers.

### Workflow

**PDFs → Text Extraction → Chunking → Embeddings → Cosine Similarity → Retrieved Evidence → Grounded Answer**

The files, extracted text, chunks, and embeddings are stored only in the current Colab runtime.

### Learning Objective

This exercise helps you understand how each component of a RAG system works before using a managed vector store.


### Step 0. Install Required Libraries

Install the Python libraries required for PDF processing, numerical calculations, and access to the OpenAI API.

In [ ]:
!pip install -q openai pypdf numpy

### Step 1. Import Required Libraries

Import the Python libraries required for file handling, PDF text extraction, numerical calculations, and OpenAI API access.

In [ ]:
import glob
import logging
import os
import re
import unicodedata

from collections import defaultdict

import numpy as np

from openai import OpenAI
from pypdf import PdfReader


# Hide repeated non-critical pypdf messages
logging.getLogger("pypdf").setLevel(logging.CRITICAL)

### Step 2. Connect to the OpenAI API

Retrieve the API key from Google Colab Secrets and use it to connect to the OpenAI API.

Make sure your API key is saved in Colab Secrets under the name `OPENAI_API_KEY`.

In [ ]:
try:
    from google.colab import userdata

    api_key = userdata.get("OPENAI_API_KEY")

except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")


if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. "
        "In Colab, open the Secrets panel, add "
        "OPENAI_API_KEY, and enable notebook access."
    )


client = OpenAI(api_key=api_key)

### Step 3. Configure the RAG System

Set the models and retrieval parameters used in this exercise.

These settings control how text is converted into embeddings, how documents are divided into chunks, and how many relevant passages are retrieved.

In [ ]:
# Model used to create embeddings
EMBED_MODEL = "text-embedding-3-small"

# Model used to generate the final answer
GEN_MODEL = "gpt-4o-mini"


# Character-based chunking settings
CHUNK_SIZE = 1800
CHUNK_OVERLAP = 250
MIN_CHUNK_SIZE = 200


# Retrieval settings
TOP_K = 10
MAX_CHUNKS_PER_PAPER = 3

# Set to False if the filter removes useful passages
FILTER_REFERENCE_MATERIAL = True

# Number of vector components to display
VECTOR_PREVIEW_SIZE = 8


# The final research question
QUESTION = (
    "Based on the uploaded papers' own empirical results, "
    "which papers report a nonlinear relationship between "
    "environmental performance and financial performance?"
)


# The retrieval query includes related expressions
# that may appear in academic papers.
RETRIEVAL_QUERY = (
    "The uploaded paper's own empirical findings or results "
    "regarding a nonlinear, U-shaped, inverted U-shaped, "
    "quadratic, or curvilinear relationship between "
    "environmental performance and financial performance. "
    "Focus on the abstract, empirical results, analysis, "
    "discussion, and conclusion rather than prior studies."
)

### Step 4. Find the Uploaded Papers

Locate the PDF files uploaded to the current Colab runtime.

Before running this cell, upload the papers using the **Files** panel on the left side of Colab.

In [ ]:
pdf_files = sorted(
    glob.glob("/content/*.pdf")
)


if not pdf_files:
    raise FileNotFoundError(
        "No PDF files were found in /content. "
        "Upload the papers using the Colab Files panel."
    )


print("=" * 80)
print("PDF FILES")
print("=" * 80)

for pdf_path in pdf_files:
    print("-", os.path.basename(pdf_path))

print("\nNumber of PDF files:", len(pdf_files))

### Step 5. Clean the Extracted PDF Text

PDF text extraction may produce formatting artifacts such as broken words, soft hyphens, and repeated spaces.

This function cleans the extracted text before it is used for retrieval.

In [ ]:
def clean_pdf_text(text):
    """
    Clean common PDF text-extraction artifacts.
    """

    # Normalize Unicode characters
    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # Remove soft hyphens
    text = text.replace(
        "\u00ad",
        ""
    )

    # Rejoin words divided across line breaks
    # Example: environ-\nmental → environmental
    text = re.sub(
        r"(?<=\w)-\s*\n\s*(?=\w)",
        "",
        text
    )

    # Collapse repeated whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

### Step 6. Detect Reference Sections

Academic papers contain reference lists that may appear semantically relevant even though they do not represent the paper's own findings.

This function identifies where the References or Bibliography section begins.

In [ ]:
def starts_reference_section(raw_text):
    """
    Check whether the beginning of a page contains
    a References or Bibliography heading.
    """

    normalized_text = unicodedata.normalize(
        "NFKC",
        raw_text
    )

    lines = [
        re.sub(r"\s+", " ", line).strip().lower()
        for line in normalized_text.splitlines()
        if line.strip()
    ]

    # Search the first 20 non-empty lines
    for line in lines[:20]:
        if re.match(
            r"^(references|bibliography)\b.{0,15}$",
            line
        ):
            return True

    return False

### Step 7. Extract Text Page by Page

Extract text from each PDF while preserving the filename, page number, and whether the page belongs to the reference section.

Keeping page-level information allows us to trace retrieved evidence back to its source.

In [ ]:
def extract_pdf_pages(pdf_path):
    """
    Extract text while preserving:
    - filename
    - page number
    - whether the page belongs to the reference section
    """

    reader = PdfReader(pdf_path)

    extracted_pages = []
    in_reference_section = False

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):
        raw_text = page.extract_text() or ""

        # Once the reference section begins,
        # subsequent pages are also marked as references.
        if starts_reference_section(raw_text):
            in_reference_section = True

        cleaned_text = clean_pdf_text(
            raw_text
        )

        if cleaned_text:
            extracted_pages.append({
                "paper": pdf_path,
                "page": page_number,
                "text": cleaned_text,
                "is_reference_page": in_reference_section
            })

    return extracted_pages


all_pages = []

for pdf_path in pdf_files:
    pages = extract_pdf_pages(
        pdf_path
    )

    all_pages.extend(
        pages
    )


if not all_pages:
    raise ValueError(
        "No text was extracted. "
        "The PDFs may contain scanned images and require OCR."
    )


print("\n" + "=" * 80)
print("TEXT EXTRACTION")
print("=" * 80)

print("Extracted pages:", len(all_pages))

print("\nSample:")
print(all_pages[0]["text"][:500])

### Step 8. Divide the Text into Chunks

Long pages are divided into smaller overlapping chunks.

Chunking allows the system to retrieve specific passages rather than processing an entire paper at once. Overlap helps preserve information that may appear near chunk boundaries.

In [ ]:
def make_chunks(
    pages,
    chunk_size,
    overlap,
    minimum_size
):
    """
    Divide each page into overlapping,
    character-based chunks.
    """

    if chunk_size <= 0:
        raise ValueError(
            "chunk_size must be greater than zero."
        )

    if not 0 <= overlap < chunk_size:
        raise ValueError(
            "overlap must be zero or greater "
            "and smaller than chunk_size."
        )

    created_chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = min(
                start + chunk_size,
                len(text)
            )

            chunk_text = text[
                start:end
            ].strip()

            if len(chunk_text) >= minimum_size:
                created_chunks.append({
                    "paper": page["paper"],
                    "page": page["page"],
                    "text": chunk_text,
                    "is_reference_page": page[
                        "is_reference_page"
                    ]
                })

            if end >= len(text):
                break

            start = end - overlap

    return created_chunks


chunks = make_chunks(
    pages=all_pages,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
    minimum_size=MIN_CHUNK_SIZE
)


if not chunks:
    raise ValueError(
        "No chunks were created. "
        "Check CHUNK_SIZE and MIN_CHUNK_SIZE."
    )


print("\n" + "=" * 80)
print("CHUNKING")
print("=" * 80)

print("Chunks:", len(chunks))
print("Chunk size:", CHUNK_SIZE, "characters")
print("Overlap:", CHUNK_OVERLAP, "characters")


print("\nSample chunk:")
print("-" * 80)

print(
    "File:",
    os.path.basename(
        chunks[0]["paper"]
    )
)

print(
    "Page:",
    chunks[0]["page"]
)

print(
    chunks[0]["text"][:500]
)

### Step 9. Create Embeddings

Convert each text chunk into a numerical vector called an embedding.

Embeddings allow texts to be compared based on semantic meaning rather than exact keyword matches.

The vectors are normalized so that their dot product can be used as cosine similarity.

In [ ]:
def create_embeddings(
    texts,
    batch_size=50
):
    """
    Convert texts into normalized embedding vectors.

    After normalization, the dot product between
    two vectors equals cosine similarity.
    """

    vectors = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):
        batch = texts[
            start:start + batch_size
        ]

        response = client.embeddings.create(
            model=EMBED_MODEL,
            input=batch
        )

        vectors.extend(
            item.embedding
            for item in response.data
        )

        completed = min(
            start + batch_size,
            len(texts)
        )

        print(
            f"Embedded {completed}/{len(texts)}"
        )

    matrix = np.asarray(
        vectors,
        dtype=np.float32
    )

    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True
    )

    matrix = matrix / np.clip(
        norms,
        1e-12,
        None
    )

    return matrix


chunk_texts = [
    chunk["text"]
    for chunk in chunks
]


print("\n" + "=" * 80)
print("CREATING EMBEDDINGS")
print("=" * 80)


embedding_matrix = create_embeddings(
    chunk_texts
)


print(
    "\nEmbedding matrix:",
    embedding_matrix.shape
)

print(
    f"First {VECTOR_PREVIEW_SIZE} dimensions:",
    np.round(
        embedding_matrix[
            0,
            :VECTOR_PREVIEW_SIZE
        ],
        6
    )
)

### Step 10. Filter Reference-Like Chunks

Some reference-list content may remain even after reference pages have been identified.

This additional heuristic looks for characteristics such as reference headings, many publication years, or repeated DOI strings.

> This is a heuristic filter and may not identify every reference-related passage correctly.

In [ ]:
def looks_like_reference_chunk(text):
    """
    Identify chunks that appear to consist mainly
    of bibliography or reference-list entries.

    This is a heuristic rather than a perfect classifier.
    """

    lower_text = text.lower().strip()

    # References or Bibliography near the beginning
    has_reference_heading = bool(
        re.search(
            r"\b(references|bibliography)\b",
            lower_text[:200]
        )
    )

    # Many publication years in one chunk
    years = re.findall(
        r"\b(?:19|20)\d{2}[a-z]?\b",
        text
    )

    # Repeated DOI strings
    doi_count = lower_text.count(
        "doi"
    )

    return (
        has_reference_heading
        or len(years) >= 12
        or doi_count >= 4
    )

### Step 11. Retrieve Relevant Evidence

Convert the retrieval query into an embedding and compare it with the embeddings of all document chunks.

Cosine similarity is used to rank the chunks according to their semantic relevance.

The system also limits the number of chunks retrieved from each paper and can exclude reference-related material.

In [ ]:
def retrieve_chunks(
    query,
    k,
    max_per_paper,
    filter_reference_material
):
    """
    Convert the query into an embedding,
    calculate cosine similarity, and retrieve
    the highest-ranking chunks.
    """

    query_response = client.embeddings.create(
        model=EMBED_MODEL,
        input=query
    )

    query_vector = np.asarray(
        query_response.data[0].embedding,
        dtype=np.float32
    )

    query_vector = query_vector / max(
        np.linalg.norm(query_vector),
        1e-12
    )

    # Because all vectors are normalized,
    # dot product equals cosine similarity.
    scores = (
        embedding_matrix
        @ query_vector
    )

    ranked_indices = np.argsort(
        scores
    )[::-1]

    selected = []
    paper_counts = defaultdict(int)

    for index in ranked_indices:
        index = int(index)

        chunk = chunks[index]

        if filter_reference_material:
            if chunk.get(
                "is_reference_page",
                False
            ):
                continue

            if looks_like_reference_chunk(
                chunk["text"]
            ):
                continue

        paper = chunk["paper"]

        if (
            paper_counts[paper]
            >= max_per_paper
        ):
            continue

        selected.append({
            "paper": paper,
            "page": chunk["page"],
            "text": chunk["text"],
            "score": float(
                scores[index]
            )
        })

        paper_counts[paper] += 1

        if len(selected) >= k:
            break

    return selected, query_vector


retrieved, query_embedding = retrieve_chunks(
    query=RETRIEVAL_QUERY,
    k=TOP_K,
    max_per_paper=MAX_CHUNKS_PER_PAPER,
    filter_reference_material=(
        FILTER_REFERENCE_MATERIAL
    )
)


if not retrieved:
    raise ValueError(
        "No chunks were retrieved. "
        "Try setting FILTER_REFERENCE_MATERIAL = False."
    )


### Step 12. Inspect the Query Embedding

The research query has also been converted into a numerical vector.

The values below show a small preview of that embedding. The full vector contains many more dimensions.

In [ ]:
print("\n" + "=" * 80)
print("QUERY EMBEDDING")
print("=" * 80)

print(
    "Vector dimensions:",
    len(query_embedding)
)

print(
    f"First {VECTOR_PREVIEW_SIZE} dimensions:",
    np.round(
        query_embedding[
            :VECTOR_PREVIEW_SIZE
        ],
        6
    )
)


### Step 13. Inspect the Retrieved Evidence

Before generating an answer, examine the passages retrieved by the system.

Check whether each passage:

- is relevant to the research question;
- represents the uploaded paper's own empirical findings;
- describes a prior study instead;
- comes from a reference section; and
- provides sufficient evidence for the intended conclusion.

> **Similarity indicates semantic relevance, not evidentiary validity.**

In [ ]:
print("\n" + "=" * 80)
print("RETRIEVED EVIDENCE")
print("=" * 80)


for source_number, item in enumerate(
    retrieved,
    start=1
):
    filename = os.path.basename(
        item["paper"]
    )

    print(
        f"\n[S{source_number}] "
        f"{filename} | "
        f"page {item['page']} | "
        f"score {item['score']:.4f}"
    )

    print("-" * 80)

    print(
        item["text"][:900]
    )



### Step 14. Build the Retrieved Context

Combine the retrieved passages into a single context that will be provided to the language model.

Each passage is assigned a source marker such as `[S1]`, `[S2]`, or `[S3]` so that the generated answer can refer back to the retrieved evidence.

In [ ]:
context_parts = []

for source_number, item in enumerate(
    retrieved,
    start=1
):
    filename = os.path.basename(
        item["paper"]
    )

    context_parts.append(
        f"[S{source_number}] "
        f"{filename}, page {item['page']}\n"
        f"{item['text']}"
    )


retrieved_context = "\n\n".join(
    context_parts
)


### Step 15. Define the Grounding Instructions

Specify how the language model should interpret and use the retrieved evidence.

The instructions require the model to distinguish:

- the paper's own empirical results;
- findings from prior studies cited in the paper; and
- other background or methodological information.

The model is also instructed not to use outside knowledge.

In [ ]:
instructions = """
You are an academic research assistant.

Answer the research question using only the retrieved
passages from the uploaded papers.

Before composing the answer, evaluate each passage as one
of the following:

A. OWN EMPIRICAL RESULT
The uploaded paper reports its own analysis or finding.

B. PRIOR STUDY
The passage reports a finding from another study cited by
the uploaded paper.

C. OTHER
The passage contains theory, background, methodology,
references, or insufficient evidence.

Follow these rules:

1. Count a paper as reporting a nonlinear relationship only
   when at least one retrieved passage clearly presents the
   uploaded paper's own empirical result.

2. Evidence may come from the abstract, results, empirical
   analysis, discussion, or conclusion, provided that it
   clearly reports the uploaded paper's own finding.

3. Do not treat a finding attributed to another author as
   the uploaded paper's own result.

4. Do not infer a finding solely from a paper's title,
   research question, theory, or literature review.

5. Distinguish a nonlinear functional relationship, such as
   a U-shaped or inverted U-shaped relationship, from a
   relationship that merely differs across time periods,
   samples, or economic conditions.

6. Cite every substantive claim using source markers such
   as [S1], [S2], or [S3].

7. Use no outside knowledge.

8. Make no claim that is unsupported by the retrieved
   passages.

9. When the retrieved passages do not contain sufficient
   evidence, state:
   "The retrieved evidence is insufficient to answer
   this question."

10. Keep the answer concise and academic.
"""


### Step 16. Generate a Grounded Answer

Send the research question together with the retrieved context and grounding instructions to the language model.

The model now generates an answer based only on the evidence retrieved from the uploaded papers.

In [ ]:
response = client.responses.create(
    model=GEN_MODEL,
    instructions=instructions,
    input=(
        f"Research question:\n"
        f"{QUESTION}\n\n"
        f"Retrieved context:\n"
        f"{retrieved_context}"
    ),
    max_output_tokens=800
)


### What Did We Build?

We created a simple in-memory RAG system that:

1. extracts text from academic papers;
2. divides the text into chunks;
3. creates embeddings;
4. retrieves semantically relevant evidence;
5. filters reference-related material; and
6. generates an evidence-grounded answer.

However, the extracted text, chunks, and embeddings exist only in the current Colab runtime.

In [ ]:
print("\n" + "=" * 80)
print("GROUNDED ANSWER")
print("=" * 80)

print(
    response.output_text
    or "[Empty output]"
)


### Next Step: Vector Stores

This exercise stored the documents and embeddings only in Colab memory.

In the next exercise, we will use an **OpenAI Vector Store** to create a document collection that can be stored, reused, and expanded with additional papers.